<a href="https://colab.research.google.com/github/satriaekad20-droid/IPL--hazard/blob/master/UTS_INF1316_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluasi Tengah Semester INF-1316 Organisasi dan Arsitektur Komputer
Dosen: Mulya Agung
## Simulator Data Hazards CPU dan Virtual Memory

**Petunjuk**
1. Isi `NIM` pada sel berikut, lalu jalankan semua sel dari atas (*Run All*).
2. **Jangan mengubah** sel generator soal dan sel konstanta. Parameter soal berbeda untuk setiap NIM.
3. Lengkapi setiap fungsi bertanda `# TODO` pada soal A-D dan **hapus baris `raise NotImplementedError`**. Jangan mengganti nama atau parameter fungsi.
4. Kode akan diuji otomatis dengan program grader menggunakan NIM Anda **dan** lima NIM lain, jadi fungsi harus bekerja untuk parameter apa pun.
5. Kumpulkan notebook ini ke eCampuz dengan nama `UTS_INF1316_<NIM>.ipynb`.

In [2]:
NIM = "51248180"

## Generator soal (jangan diubah)

In [3]:
# JANGAN DIUBAH: generator parameter soal per NIM
import random

PAGE_SIZE = 4096
SKELETONS = [  # register abstrak a..g dipetakan ke R1..R7 secara acak per NIM
    [("I1", "LOAD",  "a", [],         "m0"),
     ("I2", "ADD",   "b", ["a", "c"], None),
     ("I3", "SUB",   "a", ["d", "e"], None),
     ("I4", "STORE", None, ["b"],     "m1"),
     ("I5", "ADD",   "b", ["f", "g"], None),
     ("I6", "LOAD",  "c", [],         "m2"),
     ("I7", "STORE", None, ["c"],     "m3")],
    [("I1", "LOAD",  "a", [],         "m0"),
     ("I2", "SUB",   "b", ["c", "d"], None),
     ("I3", "ADD",   "c", ["a", "b"], None),
     ("I4", "LOAD",  "a", [],         "m1"),
     ("I5", "STORE", None, ["c"],     "m2"),
     ("I6", "ADD",   "e", ["a", "f"], None),
     ("I7", "STORE", None, ["e"],     "m3")],
    [("I1", "ADD",   "a", ["b", "c"], None),
     ("I2", "LOAD",  "b", [],         "m0"),
     ("I3", "STORE", None, ["a"],     "m1"),
     ("I4", "SUB",   "a", ["b", "d"], None),
     ("I5", "LOAD",  "c", [],         "m2"),
     ("I6", "ADD",   "e", ["a", "c"], None),
     ("I7", "STORE", None, ["e"],     "m3")],
]

def _outcome(program, pt):
    """Hitung jumlah page fault & protection fault (mode user) untuk validasi."""
    faults = prot = 0
    for _, op, _, _, addr in program:
        if addr is None: continue
        pte = pt[addr // PAGE_SIZE]
        w = op == "STORE"
        if pte["sup"] or (w and not pte["write"]) or (not w and not pte["read"]):
            prot += 1
        elif not pte["present"]:
            faults += 1
    return faults, prot

def generate(nim):
    rng = random.Random(int("".join(ch for ch in str(nim) if ch.isdigit())))
    while True:
        skeleton = rng.choice(SKELETONS)
        regs = rng.sample([f"R{i}" for i in range(1, 8)], 7)
        rmap = dict(zip("abcdefg", regs))
        vpns = rng.sample(range(4), 4)              # VPN untuk m0..m3
        addr = {f"m{k}": vpns[k] * PAGE_SIZE + rng.randrange(0, PAGE_SIZE, 4) for k in range(4)}
        program = [(n, op, rmap.get(d), [rmap[s] for s in src], addr.get(m))
                   for n, op, d, src, m in skeleton]
        frames = rng.sample(range(1, 16), 6)
        absent = set(rng.sample(range(4), 2))
        pt = {}
        for v in range(4):
            pt[v] = {"present": v not in absent, "ppn": None if v in absent else frames[v],
                     "read": 1, "write": rng.randint(0, 1), "sup": int(rng.random() < 0.25)}
        free = frames[4:6]
        f, p = _outcome(program, pt)
        if f >= 1 and p >= 1:                        # pastikan ada page fault & protection fault
            return program, pt, free

In [4]:
PAGE_FAULT_PENALTY = 100  # cycle

class PageFault(Exception): pass
class ProtectionFault(Exception): pass

PROGRAM, PAGE_TABLE, FREE_FRAMES = generate(NIM)

## Parameter soal Anda

In [5]:
print("Program:")
for name, op, dest, srcs, addr in PROGRAM:
    print(f"  {name}: {op:<5} tujuan={dest or '-':<4} sumber={', '.join(srcs) or '-':<8} alamat={hex(addr) if addr is not None else '-'}")
print("\nPage table (page size 4 KB, mode user):")
for vpn, e in PAGE_TABLE.items():
    print(f"  VPN {vpn}: present={int(e['present'])} ppn={e['ppn']} READ={e['read']} WRITE={e['write']} SUP={e['sup']}")
print("\nFREE_FRAMES =", FREE_FRAMES)

Program:
  I1: LOAD  tujuan=R1   sumber=-        alamat=0x35f4
  I2: ADD   tujuan=R4   sumber=R1, R5   alamat=-
  I3: SUB   tujuan=R1   sumber=R2, R6   alamat=-
  I4: STORE tujuan=-    sumber=R4       alamat=0x1c80
  I5: ADD   tujuan=R4   sumber=R3, R7   alamat=-
  I6: LOAD  tujuan=R5   sumber=-        alamat=0x21e8
  I7: STORE tujuan=-    sumber=R5       alamat=0xac4

Page table (page size 4 KB, mode user):
  VPN 0: present=1 ppn=10 READ=1 WRITE=0 SUP=1
  VPN 1: present=0 ppn=None READ=1 WRITE=1 SUP=0
  VPN 2: present=0 ppn=None READ=1 WRITE=1 SUP=0
  VPN 3: present=1 ppn=9 READ=1 WRITE=0 SUP=1

FREE_FRAMES = [1, 5]


## A. Deteksi data hazards (24 poin)
Kembalikan list tuple `(jenis, instr_awal, instr_akhir, register)` untuk **semua** pasangan (i, j) dengan i < j:
- **RAW**: register tujuan Ii dibaca Ij
- **WAR**: register sumber Ii ditulis Ij
- **WAW**: Ii dan Ij menulis register yang sama

In [8]:
def detect_hazards(program):
    hazards = []

    for i in range(len(program)):
        name_i, _, dest_i, srcs_i, _ = program[i]

        for j in range(i + 1, len(program)):
            name_j, _, dest_j, srcs_j, _ = program[j]

            # RAW: instruksi awal menulis register yang dibaca instruksi akhir.
            if dest_i is not None and dest_i in srcs_j:
                hazards.append(("RAW", name_i, name_j, dest_i))

            # WAR: instruksi awal membaca register yang ditulis instruksi akhir.
            if dest_j is not None and dest_j in srcs_i:
                hazards.append(("WAR", name_i, name_j, dest_j))

            # WAW: kedua instruksi menulis register tujuan yang sama.
            if dest_i is not None and dest_j is not None and dest_i == dest_j:
                hazards.append(("WAW", name_i, name_j, dest_i))

    return hazards

for h in detect_hazards(PROGRAM):
    print(h)





('RAW', 'I1', 'I2', 'R1')
('WAW', 'I1', 'I3', 'R1')
('WAR', 'I2', 'I3', 'R1')
('RAW', 'I2', 'I4', 'R4')
('WAW', 'I2', 'I5', 'R4')
('WAR', 'I2', 'I6', 'R5')
('WAR', 'I4', 'I5', 'R4')
('RAW', 'I6', 'I7', 'R5')


## B. Register renaming (18 poin)
Ganti setiap register tujuan dengan register fisik baru `P0, P1, ...` sesuai urutan, dan perbarui register sumber pada instruksi berikutnya mengikuti pemetaan terbaru.

In [9]:
def rename_registers(program):
    """Mengganti setiap tujuan dengan register fisik baru dan memperbarui sumber."""
    latest_mapping = {}
    next_physical_register = 0
    renamed_program = []

    for name, op, dest, srcs, addr in program:
        # Sumber menggunakan versi register fisik terakhir yang sudah dibuat.
        renamed_srcs = [latest_mapping.get(reg, reg) for reg in srcs]

        # Setiap penulisan memperoleh register fisik baru.
        if dest is not None:
            renamed_dest = f"P{next_physical_register}"
            latest_mapping[dest] = renamed_dest
            next_physical_register += 1
        else:
            renamed_dest = None

        renamed_program.append((name, op, renamed_dest, renamed_srcs, addr))

    return renamed_program

renamed = rename_registers(PROGRAM)
for ins in renamed:
    print(ins)
print("Hazard tersisa:", detect_hazards(renamed))

('I1', 'LOAD', 'P0', [], 13812)
('I2', 'ADD', 'P1', ['P0', 'R5'], None)
('I3', 'SUB', 'P2', ['R2', 'R6'], None)
('I4', 'STORE', None, ['P1'], 7296)
('I5', 'ADD', 'P3', ['R3', 'R7'], None)
('I6', 'LOAD', 'P4', [], 8680)
('I7', 'STORE', None, ['P4'], 2756)
Hazard tersisa: [('RAW', 'I1', 'I2', 'P0'), ('RAW', 'I2', 'I4', 'P1'), ('RAW', 'I6', 'I7', 'P4')]


## C. Virtual memory (34 poin)
**`translate`** (24 poin): hitung VPN = `vaddr // PAGE_SIZE` dan offset. Periksa berurutan:
1. `SUP = 1` dan bukan supervisor → `ProtectionFault`
2. akses `"read"` dengan `READ = 0`, atau `"write"` dengan `WRITE = 0` → `ProtectionFault`
3. `present = False` → `PageFault(vpn)`

Kembalikan alamat fisik = `PPN * PAGE_SIZE + offset`.

**`handle_page_fault`** (10 poin): ambil frame kosong pertama dari `free_frames`, tulis sebagai PPN, set `present = True`.

In [10]:
def translate(vaddr, access, page_table, supervisor=False):
    """Menerjemahkan alamat virtual ke alamat fisik sesuai hak akses halaman."""
    vpn = vaddr // PAGE_SIZE
    offset = vaddr % PAGE_SIZE
    entry = page_table[vpn]

    # Periksa proteksi supervisor terlebih dahulu.
    if entry["sup"] == 1 and not supervisor:
        raise ProtectionFault(f"VPN {vpn} hanya dapat diakses dalam mode supervisor")

    # Periksa izin baca/tulis sebelum memeriksa keberadaan halaman.
    if access == "read" and entry["read"] == 0:
        raise ProtectionFault(f"VPN {vpn} tidak mengizinkan akses baca")
    if access == "write" and entry["write"] == 0:
        raise ProtectionFault(f"VPN {vpn} tidak mengizinkan akses tulis")
    if access not in ("read", "write"):
        raise ValueError("access harus bernilai 'read' atau 'write'")

    # Halaman yang belum berada di memori utama menimbulkan page fault.
    if not entry["present"]:
        raise PageFault(vpn)

    return entry["ppn"] * PAGE_SIZE + offset


def handle_page_fault(vpn, page_table, free_frames):
    """Memetakan VPN yang belum hadir ke frame kosong pertama yang tersedia."""
    if vpn not in page_table:
        raise KeyError(f"VPN {vpn} tidak ditemukan pada page table")
    if page_table[vpn]["present"]:
        return page_table[vpn]["ppn"]
    if not free_frames:
        raise RuntimeError("Tidak ada free frame yang tersedia untuk menangani page fault")

    frame = free_frames.pop(0)
    page_table[vpn]["ppn"] = frame
    page_table[vpn]["present"] = True
    return frame

for name, op, _, _, addr in PROGRAM:
    if addr is None:
        continue
    access = "read" if op == "LOAD" else "write"
    try:
        print(name, hex(addr), "->", hex(translate(addr, access, PAGE_TABLE)))
    except PageFault as e:
        print(name, hex(addr), "-> PageFault pada VPN", e.args[0])
    except ProtectionFault as e:
        print(name, hex(addr), "-> ProtectionFault:", e)

I1 0x35f4 -> ProtectionFault: VPN 3 hanya dapat diakses dalam mode supervisor
I4 0x1c80 -> PageFault pada VPN 1
I6 0x21e8 -> PageFault pada VPN 2
I7 0xac4 -> ProtectionFault: VPN 0 hanya dapat diakses dalam mode supervisor


## D. Eksekusi dan kinerja (24 poin)
**`raw_stalls`** (8 poin): stall setiap pasangan RAW = `max(0, 3 - jarak)`, dengan jarak = selisih indeks instruksi.

**`run`** (16 poin): jalankan program, kembalikan `(total_cycles, jumlah_page_fault, total_stall, log)`.
- 1 cycle per instruksi (termasuk yang dibatalkan), +100 cycle per page fault, ditambah total stall RAW.
- Setelah page fault: tangani, lalu **ulangi** instruksi.
- Protection fault: catat di log, instruksi dibatalkan, lanjut ke instruksi berikutnya.
- **Salin** `page_table` dan `free_frames` sebelum diubah; jangan mengubah milik pemanggil.

In [11]:
def raw_stalls(program):
    """Menghitung total stall dari seluruh pasangan RAW pada program."""
    total_stall = 0

    for i in range(len(program)):
        _, _, dest_i, _, _ = program[i]
        if dest_i is None:
            continue

        for j in range(i + 1, len(program)):
            _, _, _, srcs_j, _ = program[j]
            if dest_i in srcs_j:
                distance = j - i
                total_stall += max(0, 3 - distance)

    return total_stall


def run(program, page_table, free_frames, supervisor=False):
    """Menjalankan program dan menghitung cycle, page fault, stall, serta log."""
    # Salin struktur yang akan dimodifikasi agar data milik pemanggil tetap utuh.
    local_page_table = {vpn: dict(entry) for vpn, entry in page_table.items()}
    local_free_frames = list(free_frames)

    page_fault_count = 0
    total_cycles = 0
    log = []

    for name, op, _, _, addr in program:
        # Setiap instruksi dihitung satu cycle, termasuk instruksi yang dibatalkan.
        total_cycles += 1

        # Instruksi tanpa akses memori langsung selesai dalam model sederhana ini.
        if addr is None or op not in ("LOAD", "STORE"):
            log.append(f"{name}: {op} dijalankan")
            continue

        access = "read" if op == "LOAD" else "write"

        # Instruksi yang mengalami page fault diulang setelah halaman dimuat.
        while True:
            try:
                physical_address = translate(
                    addr, access, local_page_table, supervisor=supervisor
                )
                log.append(
                    f"{name}: {op} berhasil, alamat virtual {hex(addr)} "
                    f"-> alamat fisik {hex(physical_address)}"
                )
                break
            except PageFault as error:
                vpn = error.args[0]
                page_fault_count += 1
                total_cycles += PAGE_FAULT_PENALTY
                log.append(
                    f"{name}: PageFault pada VPN {vpn}; menggunakan frame kosong "
                    "dan mengulangi instruksi"
                )
                handle_page_fault(vpn, local_page_table, local_free_frames)
            except ProtectionFault as error:
                log.append(f"{name}: ProtectionFault ({error}); instruksi dibatalkan")
                break

    total_stall = raw_stalls(program)
    total_cycles += total_stall

    return total_cycles, page_fault_count, total_stall, log

cycles, faults, stalls, log = run(PROGRAM, PAGE_TABLE, FREE_FRAMES)
for entry in log:
    print(entry)
print(f"\npage fault = {faults}, stall RAW = {stalls}, total cycle = {cycles}")

I1: ProtectionFault (VPN 3 hanya dapat diakses dalam mode supervisor); instruksi dibatalkan
I2: ADD dijalankan
I3: SUB dijalankan
I4: PageFault pada VPN 1; menggunakan frame kosong dan mengulangi instruksi
I4: STORE berhasil, alamat virtual 0x1c80 -> alamat fisik 0x1c80
I5: ADD dijalankan
I6: PageFault pada VPN 2; menggunakan frame kosong dan mengulangi instruksi
I6: LOAD berhasil, alamat virtual 0x21e8 -> alamat fisik 0x51e8
I7: ProtectionFault (VPN 0 hanya dapat diakses dalam mode supervisor); instruksi dibatalkan

page fault = 2, stall RAW = 5, total cycle = 212
